# t-SNE embeddings — C1 vs C3 (§9 v5.2 key figure)

Thin notebook: all logic lives in `sharp_har.viz.plot_embeddings`
(package code, cross-review required before freeze per §10.4 — this
notebook only calls it and saves the figure).

**Why C1 vs C3 only** (not C2, not C4): the figure's job is a
qualitative picture of representation geometry — CE vs SupCon — to
sit next to the domain-diagnostic table. C2's story is already told
by that table (no extra invariance vs C1); C4 was never trained.
Adding more panels here would be scope creep on a report that is
already tight on pages — if the team wants a C2 panel too, that is a
one-line amendment to §9, not a default.

**On TRAIN features only** (§9): the only split with every AR-set and
both environments present. No test contact.

## Environment setup

Reduced preamble (as the other diagnostics): no data staging, no GPU
needed — this runs on cached features already on Drive.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
elif (cwd.parent.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# After the clone, so the file exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
from sharp_har.utils import read_yaml

drive.mount("/content/drive")  # idempotent
CKPT_ROOT = Path(read_yaml(REPO_DIR / "configs" / "paths.yaml")["ckpt_root"])
print("Repo dir:", REPO_DIR)
print("Checkpoint root:", CKPT_ROOT)

## Locate the train-feature caches

C1 uses `best.ckpt`'s cache (already produced by the C1-lin probe
session). C3 has no `best.ckpt` (§6-C3) — its cache stem is whatever
phase B selected, read from `phase_b_selection.json`.

In [ ]:
c3_sel = json.loads((CKPT_ROOT / "C3" / "phase_b_selection.json").read_text())
c3_stem = Path(c3_sel["selected_checkpoint"]).stem
print(f"C3 selected checkpoint: {c3_stem} (val macro-F1 {c3_sel['selected_val_macro_f1']:.4f})")

FEATURE_CACHES = {
    "C1": CKPT_ROOT / "C1" / "features_best_train.npz",
    "C3": CKPT_ROOT / "C3" / f"features_{c3_stem}_train.npz",
}
for label, p in FEATURE_CACHES.items():
    assert p.exists(), f"{label}: missing {p} — run the C1-lin / C3 phase-B probe session first"
    print(label, "->", p)

## Figure

Reads: cluster compactness/purity by activity in both rows (expected
high in both — both encoders separate activities well per the probe
numbers); AR-set colors expected mixed/scattered in both rows if the
domain-diagnostic verdict holds (no readable domain in either CE or
SupCon features) — a visual echo of the diagnostic table, not a new
claim on its own (t-SNE preserves local neighborhoods, not global
structure or cluster sizes).

In [ ]:
from sharp_har.viz import plot_embeddings

fig = plot_embeddings(FEATURE_CACHES, save_path=REPO_DIR / "reports" / "embeddings_c1_vs_c3.png")
fig

## Archiving

Download the executed notebook (with output/figure) and commit it
verbatim over this file in `notebooks/diagnostics/` (rename to the
actual run date if it differs from the one in the filename), per the
folder's README. The saved PNG under `reports/` is the figure asset
for the report.